In [ ]:
import sys,os,shutil,getpass
# the work dir must be in the root of "Source"
work_dir = os.getcwd()
if 'notebooks' in work_dir :
  parent_dir = os.path.dirname(os.getcwd())
sys.path.append(parent_dir)
#print(parent_dir)

In [ ]:
# just to hide the warning messages

import warnings
warnings.filterwarnings('ignore')


In [ ]:
import altair as alt
import numpy as np
import subprocess
import folium
import random
import matplotlib.pyplot as plt
import pandas as pd
import json
import math
import numpy as np
import numpy.ma as ma
import datetime
import requests
import csv

from urllib.parse import urlparse
from folium import plugins
from netCDF4 import Dataset,num2date

from SOURCE.model_postpro import vertical_interpolation



### Climatology module
***
Extract temperature and salinity time series from climatological datasets at moorings location

#### 1. Define your time window

In [ ]:
targeted_range = '2016-01-01T00:00:00Z/2016-12-31T00:00:00Z' #set your own!

#### 2. INGV North Adriatic climatology
***

Extract temperature and salinity time series from INGV North Adriatic climatology through INGV ERDDAP 

In [ ]:
# NAdr climatology Land-sea mask paths
climdir = parent_dir+'/input_NA/CLIM/'
clim_mask_dir = parent_dir+'/input_NA/STATIC/'
clim_mask = clim_mask_dir+'/NAdr_clim_mask_bathy.nc'

if not os.path.exists(climdir):
    os.makedirs(climdir)

if not os.path.exists(clim_mask_dir):
    os.makedirs(clim_mask_dir)    

# Maximum allowed distance between mooring location and closest climatology dataset grid point (in km)
# If this distance is exceeded, climatology data will not be downloaded for the location of the mooring
Dist_max = 9  # (km) set your own!

debug = False # Set to False to silence debug output

def great_circle_distance(lat_grid, lon_grid, target_lat, target_lon):
    """
    Calculates the great-circle distance between points in a lat/lon grid
    and a single target point. Uses the spherical law of cosines.
    """
    earth_radius = 6371.0  # Earth radius in km
    
    # Convert degrees to radians
    lat_rad = np.deg2rad(lat_grid)
    lon_rad = np.deg2rad(lon_grid)
    target_lat_rad = np.deg2rad(target_lat)
    target_lon_rad = np.deg2rad(target_lon)

    # Calculate distance
    # The formula is broadcast across the entire grid
    raw_dist = np.arccos(
        np.sin(target_lat_rad) * np.sin(lat_rad) +
        np.cos(target_lat_rad) * np.cos(lat_rad) * np.cos(lon_rad - target_lon_rad)
    )
    
    return earth_radius * raw_dist


def find_closest_geo_point(lon_grid, lat_grid, target_lon, target_lat):
    """
    Finds all grid indices for point(s) closest to a target lat/lon
    using high-precision calculations.

    Args:
        lon_grid (np.ndarray): A 2D array of longitude coordinates.
        lat_grid (np.ndarray): A 2D array of latitude coordinates.
        target_lon (float): The longitude of the target point.
        target_lat (float): The latitude of the target point.

    Returns:
        tuple: A tuple containing:
               - j_indices (np.ndarray): Array of row indices for the closest point(s).
               - i_indices (np.ndarray): Array of column indices for the closest point(s).
               - min_distance (float): The distance in km to the closest point(s).
    """
    # Safeguard: Ensure all inputs are 64-bit floats for precision
    lon_grid = lon_grid.astype(np.float64)
    lat_grid = lat_grid.astype(np.float64)
    target_lon = float(target_lon)
    target_lat = float(target_lat)

    # 1. Calculate distance using the Haversine formula
    R = 6371.0  # Earth's radius in km
    lat_grid_rad, lon_grid_rad, target_lat_rad, target_lon_rad = map(np.radians, [lat_grid, lon_grid, target_lat, target_lon])

    dlon = lon_grid_rad - target_lon_rad
    dlat = lat_grid_rad - target_lat_rad

    a = np.sin(dlat / 2)**2 + np.cos(target_lat_rad) * np.cos(lat_grid_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance_matrix = R * c

    # 2. Find the single minimum distance and all points matching it
    min_distance = np.min(distance_matrix)
    j_indices, i_indices = np.where(distance_matrix == min_distance)

    # 3. Print the results for all found points
    print(f"Minimum distance found: {(min_distance * 1000):.2f} m")
    print(f"Found {len(j_indices)} point(s) at this distance:")

    for j, i in zip(j_indices, i_indices):
        lon_val = lon_grid[j, i]
        lat_val = lat_grid[j, i]
        print(f"  - Index (j={j}, i={i}) -> Coords (lon={lon_val:.6f}, lat={lat_val:.6f})")

    # 4. Return all found indices
    return j_indices, i_indices, min_distance


# Download a full-domain single surface field to create the Land-sea mask
if not os.path.exists(clim_mask):
    print("Download a full-domain single surface field file to create the Land-sea mask.")

    DATASET_ID = "NADR_CLIM_TS_3KM_1_m_emt"  # Place the correct dataset ID here
    VARIABLE_NAME = "Temperature"

    start_time = "2001-01-01T00:00:00Z"
    end_time = "2001-01-01T00:00:00Z"
    min_depth = 0.0
    max_depth = 0.0
    min_lat = 43.5
    max_lat = 46.2
    min_lon = 11.5
    max_lon = 15.982

    output_filename = clim_mask

    url = (
        f"http://oceano.bo.ingv.it/erddap/griddap/{DATASET_ID}.nc?"
        f"{VARIABLE_NAME}"
        f"[({start_time}):({end_time})]"  # Subsetting time
        f"[({min_depth}):({max_depth})]"   # Subsetting depth
        f"[({min_lat}):({max_lat})]"       
        f"[({min_lon}):({max_lon})]"    
    )

    print(f"Requesting URL: {url}")

    try:
        response = requests.get(url)
        response.raise_for_status()

        content_type = response.headers.get('Content-Type', '')
    
        if 'application/x-netcdf' in content_type:
            with open(output_filename, 'wb') as f:
                f.write(response.content)
            print(f"Download complete! File saved as: {output_filename}")
        else:
            print("Download failed. The server did not return a NetCDF file.")
            print("   Server response:")
            print(response.text)

    except requests.exceptions.RequestException as e:
        print(f"An error occurred during the request: {e}")

else:
    print("Full-domain single surface field file found.")

# Read the NAdr downloaded file
dataset = Dataset(clim_mask, 'r')
mmlatitude = dataset.variables['latitude'][:]
mmlongitude = dataset.variables['longitude'][:]
mmtemperature = dataset.variables['Temperature'][:]
dataset.close()

# Prepare the mask
mmtemperature[mmtemperature < 1e+36] = 1
mmtemperature[np.isnan(mmtemperature)] = 999999
mmmask = mmtemperature

# Create the 2D grids
lon_grid, lat_grid = np.meshgrid(mmlongitude, mmlatitude)

# Mask lon_grid and lat_grid
lon_grid_masked = lon_grid * mmmask
lon_grid_masked = np.squeeze(lon_grid_masked)
lat_grid_masked = lat_grid * mmmask
lat_grid_masked = np.squeeze(lat_grid_masked)

if debug:
    print(f"Shape of lon_grid_masked {lon_grid_masked.shape}")
    print(f"Shape of lat_grid_masked {lat_grid_masked.shape}")

# Process targeted_range     
start_datetime = targeted_range.split('/')[0][:-1]
start_month = start_datetime.split('-')[1]
print(f"Start date: {start_datetime}")
end_datetime = targeted_range.split('/')[1][:-1]
print(f"End date: {end_datetime}")
end_month = end_datetime.split('-')[1]

# Define your parameters here
DATASET_ID = "NADR_CLIM_TS_3KM_1_m_emt"

time_start = "2001-"+start_month+"-01T00:00:00Z"
time_end = "2001-"+end_month+"-01T00:00:00Z"
min_depth = 0.0
max_depth = 50.0


# Read instruments coordinates for temperature and download netcdf files for each mooring

# Observations data path
obsdir = parent_dir+'/output_NA/OBSERVATION/output/statistic/'
obsfile = os.path.join(obsdir, 'probes.csv')

with open(obsfile, 'r', newline='') as csvfile:
    reader = csv.DictReader(csvfile, delimiter=',')

    for row in reader:
            
        # Read platform code
        platform_code = row['platform_code']
        print(f"** Start processing platform {platform_code} for download")

        # Read variable_ids    
        variable = row['variable_ids']
        variable_values = row['variable_ids'].split(';')
        if debug:
            print(f"variable ids: {sorted(variable_values)}")        
        
        # Read latitude
        latitude = row['latitudes']
        latitude_values = row['latitudes'].split(';')
        if debug:
            print(f"latitude values: {sorted(latitude_values)}")
   
        # Read longitude
        longitude = row['longitudes']
        longitude_values = row['longitudes'].split(';')
        if debug:
            print(f"longitude values: {sorted(longitude_values)}")

        # Read time
        start_time = row['record_starts']      
        end_time = row['record_ends']

        counter = 0
        for var in variable_values:
            
            lat = latitude_values[counter]          
            lon = longitude_values[counter]

            # Find closest SEA grid point in NAdr climatology (distance is computed in km)
            all_j, all_i, distance = find_closest_geo_point(lon_grid_masked, lat_grid_masked, float(lon), float(lat))

            if distance > Dist_max:  # (both expressed in km)
                print(f"Closest climatology dataset grid point has been rejected because of"  + \
                        f" too large horizontal distance (> Dist_max) from mooring location")
                continue  # go to next mooring in the loop
            if all_j.size > 1:
                j_idx = all_j[0]
                i_idx = all_i[0]
                print(f"\n--> Selecting the first matching index for use: (j={j_idx}, i={i_idx})")
            elif all_j.size == 1:
                j_idx = all_j
                i_idx = all_i
                print(f"\n--> Selecting the matching index for use: (j={j_idx}, i={i_idx})")                

            closest_lat_model = float(lat_grid_masked[j_idx, i_idx])
            closest_lon_model = float(lon_grid_masked[j_idx, i_idx])
            if debug:
                print(f"closest_lat_model: {closest_lat_model}")
                print(f"closest_lon_model: {closest_lon_model}")

            if len(variable_values) == 1:
                
                # Download climatology values for temperature
                print("**************************************************************************************")
                print(f"Start downloading NAdr climatology temperature data closest to {platform_code} location:")
                VARIABLE_NAME = "Temperature"
                if debug:
                    print(f"counter: {counter}") 
                log_fileT = "download_climT_log.txt"
                
                fileout = "climnadr-data_"+platform_code+"_sea_water_temperature.nc"
                outdir = climdir+"sea_water_temperature"
                output_filename = outdir+"/"+fileout
                if not os.path.exists(outdir):
                    os.mkdir(outdir)
                
                url = (
                    f"http://oceano.bo.ingv.it/erddap/griddap/{DATASET_ID}.nc?"
                    f"{VARIABLE_NAME}"
                    f"[({time_start}):({time_end})]"
                    f"[({min_depth}):({max_depth})]"
                    f"[({closest_lat_model})]"
                    f"[({closest_lon_model})]"
                )

                print(f"Requesting URL: {url}")

                try:
                    response = requests.get(url)
                    response.raise_for_status()

                    content_type = response.headers.get('Content-Type', '')
    
                    if 'application/x-netcdf' in content_type:
                        with open(output_filename, 'wb') as f:
                            f.write(response.content)
                        print(f"Download complete! File saved as: {output_filename}")
                    else:
                        print("Download failed. The server did not return a NetCDF file.")
                        print("   Server response:")
                        print(response.text)

                except requests.exceptions.RequestException as e:
                    msg = f"Failed to download file for platform {platform_code}"
                    print(msg)
                    with open(log_fileT, 'a') as logT:
                        logT.write(msg + "\n")  
                    continue

                
                # Download climatology values for salinity
                print("**************************************************************************************")
                print(f"Start downloading NAdr climatology salinity data closest to {platform_code} location:")
                VARIABLE_NAME = "Salinity"
                if debug:
                    print(f"counter: {counter}") 
                log_fileT = "download_climS_log.txt"
                
                fileout = "climnadr-data_"+platform_code+"_sea_water_practical_salinity.nc"
                outdir = climdir+"sea_water_practical_salinity"
                output_filename = outdir+"/"+fileout
                if not os.path.exists(outdir):
                    os.mkdir(outdir)
                
                url = (
                    f"http://oceano.bo.ingv.it/erddap/griddap/{DATASET_ID}.nc?"
                    f"{VARIABLE_NAME}"
                    f"[({time_start}):({time_end})]"
                    f"[({min_depth}):({max_depth})]"
                    f"[({closest_lat_model})]"
                    f"[({closest_lon_model})]"
                )

                print(f"Requesting URL: {url}")

                try:
                    response = requests.get(url)
                    response.raise_for_status()

                    content_type = response.headers.get('Content-Type', '')
    
                    if 'application/x-netcdf' in content_type:
                        with open(output_filename, 'wb') as f:
                            f.write(response.content)
                        print(f"Download complete! File saved as: {output_filename}")
                    else:
                        print("Download failed. The server did not return a NetCDF file.")
                        print("   Server response:")
                        print(response.text)

                except requests.exceptions.RequestException as e:
                    msg = f"Failed to download file for platform {platform_code}"
                    print(msg)
                    with open(log_fileS, 'a') as logS:
                        logS.write(msg + "\n")  
                    continue

            elif len(variable_values) == 2:   
                
                if var == '1':

                    # Download climatology values for temperature
                    print("**************************************************************************************")
                    print(f"Start downloading NAdr climatology temperature data closest to {platform_code} location:")
                    VARIABLE_NAME = "Temperature"
                    if debug:
                        print(f"counter: {counter}") 
                    log_fileT = "download_climT_log.txt"
                
                    fileout = "climnadr-data_"+platform_code+"_sea_water_temperature.nc"
                    outdir = climdir+"sea_water_temperature"
                    output_filename = outdir+"/"+fileout
                    if not os.path.exists(outdir):
                        os.mkdir(outdir)
                
                    url = (
                        f"http://oceano.bo.ingv.it/erddap/griddap/{DATASET_ID}.nc?"
                        f"{VARIABLE_NAME}"
                        f"[({time_start}):({time_end})]"
                        f"[({min_depth}):({max_depth})]"
                        f"[({closest_lat_model})]"
                        f"[({closest_lon_model})]"
                    )

                    print(f"Requesting URL: {url}")

                    try:
                        response = requests.get(url)
                        response.raise_for_status()

                        content_type = response.headers.get('Content-Type', '')
    
                        if 'application/x-netcdf' in content_type:
                            with open(output_filename, 'wb') as f:
                                f.write(response.content)
                            print(f"Download complete! File saved as: {output_filename}")
                        else:
                            print("Download failed. The server did not return a NetCDF file.")
                            print("   Server response:")
                            print(response.text)

                    except requests.exceptions.RequestException as e:
                        msg = f"Failed to download file for platform {platform_code}"
                        print(msg)
                        with open(log_fileT, 'a') as logT:
                            logT.write(msg + "\n")  
                        continue          

                elif var == '2':

                    # Download climatology values for salinity
                    print("**************************************************************************************")
                    print(f"Start downloading NAdr climatology salinity data closest to {platform_code} location:")
                    VARIABLE_NAME = "Salinity"
                    if debug:
                        print(f"counter: {counter}") 
                    log_fileT = "download_climS_log.txt"
                
                    fileout = "climnadr-data_"+platform_code+"_sea_water_practical_salinity.nc"
                    outdir = climdir+"sea_water_practical_salinity"
                    output_filename = outdir+"/"+fileout
                    if not os.path.exists(outdir):
                        os.mkdir(outdir)
                
                    url = (
                        f"http://oceano.bo.ingv.it/erddap/griddap/{DATASET_ID}.nc?"
                        f"{VARIABLE_NAME}"
                        f"[({time_start}):({time_end})]"
                        f"[({min_depth}):({max_depth})]"
                        f"[({closest_lat_model})]"
                        f"[({closest_lon_model})]"
                    )

                    print(f"Requesting URL: {url}")

                    try:
                        response = requests.get(url)
                        response.raise_for_status()

                        content_type = response.headers.get('Content-Type', '')
    
                        if 'application/x-netcdf' in content_type:
                            with open(output_filename, 'wb') as f:
                                f.write(response.content)
                            print(f"Download complete! File saved as: {output_filename}")
                        else:
                            print("Download failed. The server did not return a NetCDF file.")
                            print("   Server response:")
                            print(response.text)

                    except requests.exceptions.RequestException as e:
                        msg = f"Failed to download file for platform {platform_code}"
                        print(msg)
                        with open(log_fileS, 'a') as logS:
                            logS.write(msg + "\n")  
                        continue   

            else:
                    msg = f"Unexpected number of variable ids for platform {platform_code}"
                    print(msg)
                    with open(log_fileT, 'a') as logT:
                        logT.write(msg + "\n") 
                    with open(log_fileS, 'a') as logS:
                        logS.write(msg + "\n")                        
                    continue   

            counter += 1        
                    

***

Vertically interpolate temperature and salinity time series on mooring's depths and derive daily values from monthly climatological values for the period defined by targeted_range variable

In [ ]:
# Vertically interpolate temperature and salinity time series on mooring's depths
# Use SOURCE function vertical_interpolation(in_file, depth_array_str, out_file, verbose)
# => vertical_interpolation.vertical_interpolation(location_ported_file, depth_array_str,
#                                                  vertical_interpolated_file, verbose=verbose)

data_type = 'MO'
frequency = 'dm'

# Define paths
outobsdir = parent_dir+'/output_NA/OBSERVATION/output/statistic/'+frequency
outobsdirT = outobsdir+'/sea_water_temperature/'
outobsdirS = outobsdir+'/sea_water_practical_salinity/'

inclimdirT = climdir+'/sea_water_temperature/'
inclimdirS = climdir+'/sea_water_practical_salinity/'
workclimdir = parent_dir+'/output_NA/CLIM/output/work/'

if not os.path.exists(workclimdir):
    os.makedirs(workclimdir)    

# Temperature
print(f"**== Processing temperature ==**")
sorted_files_obsT = sorted([ x for x in os.listdir(outobsdirT) if x.endswith('nc')])  
sorted_files_climT = sorted([ x for x in os.listdir(inclimdirT) if x.endswith('nc') and x.startswith('climnadr')])  

for f in sorted_files_obsT:
    mooring = f.split('_')[1]
    print(f"** Reading Platform: {mooring}")
    try:
        clim_file = [clim_f for clim_f in sorted_files_climT if clim_f.split('_')[1] == mooring][0]
        print(f"Corresponding clim file: {clim_file}")

        obs_nc = Dataset(outobsdirT+'/'+f,'r')
        obs_depth = obs_nc.variables['depth'][:]
        if debug:
            print(f"obs_depth: {obs_depth}")
        obs_depth_str = ' '.join([str(value) for value in obs_depth])

        clim_nc = Dataset(inclimdirT+'/'+clim_file,'r')
        clim_depth = clim_nc.variables['depth'][:]
        if debug:
            print(f"clim_depth: {clim_depth}")

        infileprefix = clim_file.split('-')[0]
        infilesuffix = '-'.join(clim_file.split('-')[1:])
        outfileprefix = "climnadrint"
        outfilename =  outfileprefix + '-' + infilesuffix
        print(f"Output file name: {outfilename}")

        vertical_interpolation.vertical_interpolation(inclimdirT+clim_file, obs_depth_str,
                                                      workclimdir+outfilename, True)
    except IndexError as e:
        msg = f"Climatological temperature file not found for platform {mooring}"
        print(msg)
        continue


# Salinity
print(f"**== Processing salinity ==**")
sorted_files_obsS = sorted([ x for x in os.listdir(outobsdirS) if x.endswith('nc')])  
sorted_files_climS = sorted([ x for x in os.listdir(inclimdirS) if x.endswith('nc') and x.startswith('climnadr')])

for f in sorted_files_obsS:
    mooring = f.split('_')[1]
    print(f"** Reading Platform: {mooring}")
    try:
        clim_file = [clim_f for clim_f in sorted_files_climS if clim_f.split('_')[1] == mooring][0]
        print(f"Corresponding clim file: {clim_file}")

        obs_nc = Dataset(outobsdirS+'/'+f,'r')
        obs_depth = obs_nc.variables['depth'][:]
        if debug:
            print(f"obs_depth: {obs_depth}")
        obs_depth_str = ' '.join([str(value) for value in obs_depth])

        clim_nc = Dataset(inclimdirS+'/'+clim_file,'r')
        clim_depth = clim_nc.variables['depth'][:]
        if debug:
            print(f"clim_depth: {clim_depth}")

        infileprefix = clim_file.split('-')[0]
        infilesuffix = '-'.join(clim_file.split('-')[1:])
        outfileprefix = "climnadrint"
        outfilename =  outfileprefix + '-' + infilesuffix
        print(f"Output file name: {outfilename}")

        vertical_interpolation.vertical_interpolation(inclimdirS+clim_file, obs_depth_str,
                                                  workclimdir+outfilename, True)
    except IndexError as e:
        msg = f"Climatological salinity file not found for platform {mooring}"
        print(msg)
        continue    
        

***

Derive daily values from monthly climatological values for the period defined by targeted_range variable

In [ ]:
from datetime import datetime, timedelta

debug = False

outclimdir = parent_dir+'/output_NA/CLIM/output/statistic/'+frequency+'/'
outclimdirT = outclimdir+'/sea_water_temperature/'
outclimdirS = outclimdir+'/sea_water_practical_salinity/'

if not os.path.exists(outclimdirT):
    os.makedirs(outclimdirT)

if not os.path.exists(outclimdirS):
    os.makedirs(outclimdirS)

def derive_daily_values_constant(targeted_range, monthly_values):
    """
    Derives daily values from monthly climatological values using a constant
    assignment method.

    Args:
        targeted_range (str): A string representing the date range in the format
                              'YYYY-MM-DDTHH:MM:SSZ/YYYY-MM-DDTHH:MM:SSZ'.
        monthly_values (dict): A dictionary where keys are month numbers (1-12)
                               and values are the climatological values for that month.

    Returns:
        pandas.DataFrame: A DataFrame with two columns: 'date' and 'value',
                          containing the daily derived values.
    """
    # --- 1. Parse the targeted date range ---
    try:
        start_date_str, end_date_str = targeted_range.split('/')
        # Parse the ISO 8601 format, ignoring the 'Z' for UTC
        start_date = datetime.fromisoformat(start_date_str.replace('Z', '+00:00'))
        #print(f"Start date {start_date}.")
        end_date = datetime.fromisoformat(end_date_str.replace('Z', '+00:00'))
        #print(f"End date {end_date}.")        
    except ValueError as e:
        print(f"Error parsing date range: {e}")
        return None

    # --- 2. Generate a list of dates for the period ---
    # Use pandas date_range for convenience
    daily_dates = pd.date_range(start=start_date, end=end_date, freq='D')

    # --- 3. Assign monthly values to each day ---
    derived_data = []
    for date in daily_dates:
        month = date.month
        # Look up the value for the current month
        value = monthly_values.get(month)
        #print(f"value: {value}.")

        if value is not None:
            derived_data.append({'date': date, 'value': value})
        else:
            # Handle cases where a month's data might be missing
            print(f"Warning: No value found for month {month}. Skipping date {date.date()}.")


    # --- 4. Create and return a pandas DataFrame ---
    if not derived_data:
        return pd.DataFrame(columns=['date', 'value'])

    return pd.DataFrame(derived_data)


print(f"Deriving daily values for the period: {targeted_range}\n")

sorted_files_climT = sorted([ x for x in os.listdir(workclimdir) if x.endswith('temperature.nc') and x.startswith('climnadrint')])
sorted_files_climS = sorted([ x for x in os.listdir(workclimdir) if x.endswith('salinity.nc') and x.startswith('climnadrint')])


# Temperature
print(f"==== Processing temperature data ====")
for f in sorted_files_climT:
    mooring = f.split('_')[1]
    print(f"** Reading Platform: {mooring}")

    clim_nc = Dataset(workclimdir+'/'+f,'r')
    source_global_attrs = {attr: clim_nc.getncattr(attr) for attr in clim_nc.ncattrs()}
    clim_var = clim_nc.variables['Temperature'][:]
    clim_depth = clim_nc.variables['depth'][:]
    # 1. Get the time variable object from the file
    clim_time = clim_nc.variables['time']
    # 2. Now, get the numerical values from the object.
    clim_time_values = clim_time[:]
    # 3. Call num2date using the attributes from the variable object.
    datetime_objects = num2date(clim_time_values, 
                            units=clim_time.units, 
                            calendar=clim_time.calendar)
    # 4. Loop through the datetime objects to format and extract the month
    month_list = []
    for dt in datetime_objects:
        # Format the date as 'YYYY-MM-DD'
        formatted_date = dt.strftime('%Y-%m-%d')
        
        # Extract the month value
        month_value = dt.month

        # Append the extracted month to a list
        month_list.append(month_value)
        
        print(f"Original datetime: {dt}, Formatted Date: {formatted_date}, Extracted Month: {month_value}")
    
    clim_var_dict = {}
    for i in range(len(month_list)):
        month_number = month_list[i]  # This will be 11
        temperature_profile = clim_var[i]
        clim_var_dict[month_number] = temperature_profile

    daily_values_df = derive_daily_values_constant(targeted_range, clim_var_dict)

    # Add a column with time in seconds since epoch ---
    # Define the Unix epoch start time (timezone-aware)
    epoch_start = pd.Timestamp("1970-01-01", tz='UTC')
    # Add 12 hours to the dates and then convert into seconds
    daily_values_df['date'] = daily_values_df['date'] + pd.Timedelta(hours=12)
    daily_values_df['time_seconds'] = (daily_values_df['date'].dt.tz_convert('UTC') - epoch_start).dt.total_seconds().astype(float)
    
    if debug: 
        # Set pandas to display all rows to see the full result
        pd.set_option('display.max_rows', None)
                
        print("--- Derived Daily Temperature Values (with seconds from epoch) ---")
        print(daily_values_df.to_string(
            formatters={'time_seconds': '{:.0f}'.format},
            index=False
        ))
        print("----------------------------------------------------------------")

    
    # --- Write to NetCDF file ---
    infileclim = workclimdir+'/'+f
    print(f"Input clim file: {infileclim}")
    infileprefix = infileclim.split('-')[0]
    infilesuffix = '-'.join(infileclim.split('-')[1:])
    outfileprefix = "climnadrdaily"
    output_nc_filename = outclimdirT + outfileprefix + '-' + infilesuffix
    print(f"Output clim file: {output_nc_filename}")

    with Dataset(output_nc_filename, 'w', format='NETCDF4') as ncfile:
        # --- Global Attributes ---
        # Copy attributes from the source file
        for attr_name, attr_value in source_global_attrs.items():
            ncfile.setncattr(attr_name, attr_value)
        # Add global attributes to the processed files
        ncfile.SOURCE_institution = 'Istituto Nazionale di Geofisica e Vulcanologia - Bologna, Italy'
        ncfile.SOURCE_platform_code = mooring
        ncfile.SOURCE_variable_type = 'Daily values derived from monthly climatologies'
        ncfile.SOURCE_field_type = 'Vertical interpolated climatology data'
                       
        # --- Dimensions ---
        time_len = len(daily_values_df)
        depth_len = len(clim_depth)
        ncfile.createDimension('time', time_len)
        ncfile.createDimension('depth', depth_len)

        # --- Variables and their attributes ---
        time = ncfile.createVariable('time', 'f8', ('time',))
        time.units = 'seconds since 1970-01-01 00:00:00 UTC'
        time[:] = daily_values_df['time_seconds'].values

        depth = ncfile.createVariable('depth', 'f4', ('depth',))
        depth.units = 'meters'
        depth[:] = clim_depth
                
        temp = ncfile.createVariable('sea_water_temperature', 'f4', ('time', 'depth'))
        temp.units = 'celsius'
        temp.long_name = 'Sea temperature'
                
        final_temp_data = np.vstack(daily_values_df['value'].values)
        temp[:, :] = final_temp_data

    print("NetCDF file created successfully.")



# Salinity
print(f"==== Processing salinity data ====")
for f in sorted_files_climS:
    mooring = f.split('_')[1]
    print(f"** Reading Platform: {mooring}")

    clim_nc = Dataset(workclimdir+'/'+f,'r')
    source_global_attrs = {attr: clim_nc.getncattr(attr) for attr in clim_nc.ncattrs()}
    clim_var = clim_nc.variables['Salinity'][:]
    clim_depth = clim_nc.variables['depth'][:]
    # 1. Get the time variable object from the file
    clim_time = clim_nc.variables['time']
    # 2. Now, get the numerical values from the object.
    clim_time_values = clim_time[:]
    # 3. Call num2date using the attributes from the variable object.
    datetime_objects = num2date(clim_time_values, 
                            units=clim_time.units, 
                            calendar=clim_time.calendar)
    # 4. Loop through the datetime objects to format and extract the month
    month_list = []
    for dt in datetime_objects:
        # Format the date as 'YYYY-MM-DD'
        formatted_date = dt.strftime('%Y-%m-%d')
        
        # Extract the month value
        month_value = dt.month

        # Append the extracted month to a list
        month_list.append(month_value)
        
        print(f"Original datetime: {dt}, Formatted Date: {formatted_date}, Extracted Month: {month_value}")

    clim_var_dict = {}
    for i in range(len(month_list)):
        month_number = month_list[i]  # This will be 11
        salinity_profile = clim_var[i]
        clim_var_dict[month_number] = salinity_profile

    daily_values_df = derive_daily_values_constant(targeted_range, clim_var_dict)

    # Add a column with time in seconds since epoch ---
    # Define the Unix epoch start time (timezone-aware)
    epoch_start = pd.Timestamp("1970-01-01", tz='UTC')
    # Add 12 hours to the dates and then convert into seconds
    daily_values_df['date'] = daily_values_df['date'] + pd.Timedelta(hours=12)
    daily_values_df['time_seconds'] = (daily_values_df['date'].dt.tz_convert('UTC') - epoch_start).dt.total_seconds().astype(float)
    
    if debug: 
        # Set pandas to display all rows to see the full result
        pd.set_option('display.max_rows', None)
                
        print("--- Derived Daily Salinity Values (with seconds from epoch) ---")
        print(daily_values_df.to_string(
            formatters={'time_seconds': '{:.0f}'.format},
            index=False
        ))
        print("----------------------------------------------------------------")

    
    # --- Write to NetCDF file ---
    infileclim = workclimdir+'/'+f
    print(f"Input clim file: {infileclim}")
    infileprefix = infileclim.split('-')[0]
    infilesuffix = '-'.join(infileclim.split('-')[1:])
    outfileprefix = "climnadrdaily"
    output_nc_filename = outclimdirS + outfileprefix + '-' + infilesuffix
    print(f"Output clim file: {output_nc_filename}")

    with Dataset(output_nc_filename, 'w', format='NETCDF4') as ncfile:
        # --- Global Attributes ---
        # Copy attributes from the source file
        for attr_name, attr_value in source_global_attrs.items():
            ncfile.setncattr(attr_name, attr_value)
        # Add global attributes to the processed files
        ncfile.SOURCE_institution = 'Istituto Nazionale di Geofisica e Vulcanologia - Bologna, Italy'
        ncfile.SOURCE_platform_code = mooring
        ncfile.SOURCE_variable_type = 'Daily values derived from monthly climatologies'
        ncfile.SOURCE_field_type = 'Vertical interpolated climatology data'
                       
        # --- Dimensions ---
        time_len = len(daily_values_df)
        depth_len = len(clim_depth)
        ncfile.createDimension('time', time_len)
        ncfile.createDimension('depth', depth_len)

        # --- Variables and their attributes ---
        time = ncfile.createVariable('time', 'f8', ('time',))
        time.units = 'seconds since 1970-01-01 00:00:00 UTC'
        time[:] = daily_values_df['time_seconds'].values

        depth = ncfile.createVariable('depth', 'f4', ('depth',))
        depth.units = 'meters'
        depth[:] = clim_depth
                
        salt = ncfile.createVariable('sea_water_practical_salinity', 'f4', ('time', 'depth'))
        salt.units = '"0.001'
        salt.long_name = 'Practical salinity'
                
        final_salt_data = np.vstack(daily_values_df['value'].values)
        salt[:, :] = final_salt_data

    print("NetCDF file created successfully.")
    

#### 3. ULIEGE North Adriatic climatology
***

Extract temperature and salinity time series from ULIEGE 1/16 North Adriatic climatology locally stored 

In [ ]:
from datetime import datetime, timedelta

# Uliege climatology paths
uliegedir = parent_dir+'/climfiles/uliege_V2/'
uliegemask = uliegedir+'/uliege_V2_clim_mask_bathy.nc'

inclimdir = parent_dir+'/input_NA/CLIM/'
inclimdirT = inclimdir+'/sea_water_temperature/'
inclimdirS = inclimdir+'/sea_water_practical_salinity/'

if not os.path.exists(inclimdirT):
    os.makedirs(inclimdirT)    

if not os.path.exists(inclimdirS):
    os.makedirs(inclimdirS)

    
# Maximum allowed distance between mooring location and closest climatology dataset grid point (in km)
# If this distance is exceeded, climatology data will not be downloaded for the location of the mooring
Dist_max = 9  # (km) set your own!

debug = False # Set to False to silence debug output

def great_circle_distance(lat_grid, lon_grid, target_lat, target_lon):
    """
    Calculates the great-circle distance between points in a lat/lon grid
    and a single target point. Uses the spherical law of cosines.
    """
    earth_radius = 6371.0  # Earth radius in km
    
    # Convert degrees to radians
    lat_rad = np.deg2rad(lat_grid)
    lon_rad = np.deg2rad(lon_grid)
    target_lat_rad = np.deg2rad(target_lat)
    target_lon_rad = np.deg2rad(target_lon)

    # Calculate distance
    # The formula is broadcast across the entire grid
    raw_dist = np.arccos(
        np.sin(target_lat_rad) * np.sin(lat_rad) +
        np.cos(target_lat_rad) * np.cos(lat_rad) * np.cos(lon_rad - target_lon_rad)
    )
    
    return earth_radius * raw_dist


def find_closest_geo_point(lon_grid, lat_grid, target_lon, target_lat):
    """
    Finds all grid indices for point(s) closest to a target lat/lon
    using high-precision calculations.

    Args:
        lon_grid (np.ndarray): A 2D array of longitude coordinates.
        lat_grid (np.ndarray): A 2D array of latitude coordinates.
        target_lon (float): The longitude of the target point.
        target_lat (float): The latitude of the target point.

    Returns:
        tuple: A tuple containing:
               - j_indices (np.ndarray): Array of row indices for the closest point(s).
               - i_indices (np.ndarray): Array of column indices for the closest point(s).
               - min_distance (float): The distance in km to the closest point(s).
    """
    # Safeguard: Ensure all inputs are 64-bit floats for precision
    lon_grid = lon_grid.astype(np.float64)
    lat_grid = lat_grid.astype(np.float64)
    target_lon = float(target_lon)
    target_lat = float(target_lat)

    # 1. Calculate distance using the Haversine formula
    R = 6371.0  # Earth's radius in km
    lat_grid_rad, lon_grid_rad, target_lat_rad, target_lon_rad = map(np.radians, [lat_grid, lon_grid, target_lat, target_lon])

    dlon = lon_grid_rad - target_lon_rad
    dlat = lat_grid_rad - target_lat_rad

    a = np.sin(dlat / 2)**2 + np.cos(target_lat_rad) * np.cos(lat_grid_rad) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    distance_matrix = R * c

    # 2. Find the single minimum distance and all points matching it
    min_distance = np.min(distance_matrix)
    j_indices, i_indices = np.where(distance_matrix == min_distance)

    # 3. Print the results for all found points
    print(f"Minimum distance found: {(min_distance * 1000):.2f} m")
    print(f"Found {len(j_indices)} point(s) at this distance:")

    for j, i in zip(j_indices, i_indices):
        lon_val = lon_grid[j, i]
        lat_val = lat_grid[j, i]
        print(f"  - Index (j={j}, i={i}) -> Coords (lon={lon_val:.6f}, lat={lat_val:.6f})")

    # 4. Return all found indices
    return j_indices, i_indices, min_distance


# Read temperature climatology
dataset = Dataset(uliegedir+'North_Adriatic_sea_water_temperature_V2.nc','r')
mmlatitude = dataset.variables['lat'][:]
mmlongitude = dataset.variables['lon'][:]
mmdepth = dataset.variables['depth'][:]
mm3dtemperature = dataset.variables['sea_water_temperature'][:]
mmtime = dataset.variables['time'][:]
time_var = dataset.variables['time']
time_units = time_var.units
calendar = getattr(time_var, 'calendar', 'standard')
source_global_attrs_T = {attr: dataset.getncattr(attr) for attr in dataset.ncattrs()}    
dataset.close()

# Convert time from days since 1900-01-01 00:00:00 to seconds since 1970-01-01 00:00:00 UTC
dates = num2date(mmtime, units=time_units, calendar=calendar)
epoch = datetime(1970, 1, 1)
seconds_since_1970 = [(dt - epoch).total_seconds() for dt in dates]

if debug:
    print(f"time: {seconds_since_1970}")

# Read salinity climatology
dataset = Dataset(uliegedir+'North_Adriatic_sea_water_salinity_V2.nc','r')
mm3dsalinity = dataset.variables['sea_water_salinity'][:]
source_global_attrs_S = {attr: dataset.getncattr(attr) for attr in dataset.ncattrs()} 
dataset.close()

# Create the Land-sea mask from a surface field
mmtemperature = mm3dtemperature[0, 0, :, :]
if debug:
    print(f"Shape of mmtemperature {mmtemperature.shape}")

# Prepare the mask
mmtemperature[mmtemperature < 1e+36] = 1
mmtemperature[np.isnan(mmtemperature)] = 999999
mmmask = mmtemperature

# Create the 2D grids
lon_grid, lat_grid = np.meshgrid(mmlongitude, mmlatitude)

# Mask lon_grid and lat_grid
lon_grid_masked = lon_grid * mmmask
lon_grid_masked = np.squeeze(lon_grid_masked)
lat_grid_masked = lat_grid * mmmask
lat_grid_masked = np.squeeze(lat_grid_masked)

if debug:
    print(f"Shape of lon_grid_masked {lon_grid_masked.shape}")
    print(f"Shape of lat_grid_masked {lat_grid_masked.shape}")
    
# Process targeted_range     
start_datetime = targeted_range.split('/')[0][:-1]
start_month = start_datetime.split('-')[1]
print(f"Start date: {start_datetime}")
end_datetime = targeted_range.split('/')[1][:-1]
print(f"End date: {end_datetime}")
end_month = end_datetime.split('-')[1]

# Read instruments coordinates for temperature and download netcdf files for each mooring

# Observations data path
obsdir = parent_dir+'/output_NA/OBSERVATION/output/statistic/'
obsfile = os.path.join(obsdir, 'probes.csv')

with open(obsfile, 'r', newline='') as csvfile:
    reader = csv.DictReader(csvfile, delimiter=',')

    for row in reader:
            
        # Read platform code
        platform_code = row['platform_code']
        print(f"** Start processing platform {platform_code} for download")

        # Read variable_ids    
        variable = row['variable_ids']
        variable_values = row['variable_ids'].split(';')
        if debug:
            print(f"variable ids: {sorted(variable_values)}")        
        
        # Read latitude
        latitude = row['latitudes']
        latitude_values = row['latitudes'].split(';')
        if debug:
            print(f"latitude values: {sorted(latitude_values)}")
   
        # Read longitude
        longitude = row['longitudes']
        longitude_values = row['longitudes'].split(';')
        if debug:
            print(f"longitude values: {sorted(longitude_values)}")

        # Read time
        start_time = row['record_starts']      
        end_time = row['record_ends']

        counter = 0
        for var in variable_values:
           
            lat = latitude_values[counter]          
            lon = longitude_values[counter]

            # Find closest SEA grid point in Uliege climatology (distance is computed in km)
            all_j, all_i, distance = find_closest_geo_point(lon_grid_masked, lat_grid_masked, float(lon), float(lat))

            if distance > Dist_max:  # (both expressed in km)
                print(f"Closest climatology dataset grid point has been rejected because of"  + \
                        f" too large horizontal distance (> Dist_max) from mooring location")
                continue  # go to next mooring in the loop
            if all_j.size > 1:
                j_idx = all_j[0]
                i_idx = all_i[0]
                print(f"\n--> Selecting the first matching index for use: (j={j_idx}, i={i_idx})")
            elif all_j.size == 1:
                j_idx = all_j
                i_idx = all_i
                print(f"\n--> Selecting the matching index for use: (j={j_idx}, i={i_idx})")                

            closest_lat_model = float(lat_grid_masked[j_idx, i_idx])
            closest_lon_model = float(lon_grid_masked[j_idx, i_idx])
            if debug:
                print(f"closest_lat_model: {closest_lat_model}")
                print(f"closest_lon_model: {closest_lon_model}")
            
            if len(variable_values) == 1:
                
                # Extract climatology values for temperature
                print("**************************************************************************************")
                print(f"Start extracting Uliege climatology temperature data closest to {platform_code} location:")

                if debug:
                    print(f"counter: {counter}") 
                
                fileout = "climuliege-data_"+platform_code+"_sea_water_temperature.nc"
                outdir = inclimdirT
                output_filename = outdir+"/"+fileout

                tindex_start = int(start_month) - 1
                tindex_end = int(end_month)
                if debug:
                    print(f"tindex_start: {tindex_start}")
                    print(f"tindex_end: {tindex_end}")                   
                
                closestT = mm3dtemperature[tindex_start:tindex_end, :, j_idx, i_idx]
                unixtime = seconds_since_1970[tindex_start:tindex_end]
                
                if debug:
                    print(f"Shape of closestT {closestT.shape}")

                # --- Write to NetCDF file ---
                print(f"Creating: {output_filename}")
 
                with Dataset(output_filename, 'w', format='NETCDF4') as ncfile:
                    # --- Global Attributes ---
                    # Copy attributes from the source file
                    for attr_name, attr_value in source_global_attrs_T.items():
                        ncfile.setncattr(attr_name, attr_value)
                       
                    # --- Dimensions ---
                    time_len = tindex_end - tindex_start
                    if debug:
                        print(f"time_len: {time_len}")
                    depth_len = len(mmdepth)
                    latitude_len = 1
                    longitude_len = 1
                    ncfile.createDimension('time', time_len)
                    ncfile.createDimension('depth', depth_len)
                    ncfile.createDimension('latitude', latitude_len)
                    ncfile.createDimension('longitude', longitude_len)
                    
                    # --- Variables and their attributes ---
                    time = ncfile.createVariable('time', 'f8', ('time',))
                    time.units = 'seconds since 1970-01-01 00:00:00 UTC'
                    time.calendar = 'standard'
                    if debug:
                        print(f"Shape of time[:] {time[:].shape}")
                        print(f"Shape of unixtime {len(unixtime)}")
                    time[:] = unixtime

                    depth = ncfile.createVariable('depth', 'f4', ('depth',))
                    depth.units = 'meters'
                    depth[:] = mmdepth

                    latitude = ncfile.createVariable('latitude', 'f4', ('latitude',))
                    latitude.units = 'degrees_north'
                    latitude[:] = closest_lat_model

                    longitude = ncfile.createVariable('longitude', 'f4', ('longitude',))
                    longitude.units = 'degrees_east'
                    longitude[:] = closest_lon_model                    
                
                    temp = ncfile.createVariable('Temperature', 'f4', ('time', 'depth', 'latitude', 'longitude',))
                    temp.units = 'celsius'
                    temp.long_name = 'Sea temperature'
                    
                    final_temp_data = np.vstack(closestT)
                    temp[:, :] = final_temp_data

                print("NetCDF file created successfully.")


                # Extract climatology values for salinity
                print("**************************************************************************************")
                print(f"Start extracting Uliege climatology salinity data closest to {platform_code} location:")

                if debug:
                    print(f"counter: {counter}") 
                
                fileout = "climuliege-data_"+platform_code+"_sea_water_practical_salinity.nc"
                outdir = inclimdirS
                output_filename = outdir+"/"+fileout

                tindex_start = int(start_month) - 1
                tindex_end = int(end_month)
                if debug:
                    print(f"tindex_start: {tindex_start}")
                    print(f"tindex_end: {tindex_end}")                   
                
                closestS = mm3dsalinity[tindex_start:tindex_end, :, j_idx, i_idx]
                unixtime = seconds_since_1970[tindex_start:tindex_end]
                
                if debug:
                    print(f"Shape of closestS {closestS.shape}")

                # --- Write to NetCDF file ---
                print(f"Creating: {output_filename}")
 
                with Dataset(output_filename, 'w', format='NETCDF4') as ncfile:
                    # --- Global Attributes ---
                    # Copy attributes from the source file
                    for attr_name, attr_value in source_global_attrs_S.items():
                        ncfile.setncattr(attr_name, attr_value)
                       
                    # --- Dimensions ---
                    time_len = tindex_end - tindex_start
                    if debug:
                        print(f"time_len: {time_len}")
                    depth_len = len(mmdepth)
                    latitude_len = 1
                    longitude_len = 1
                    ncfile.createDimension('time', time_len)
                    ncfile.createDimension('depth', depth_len)
                    ncfile.createDimension('latitude', latitude_len)
                    ncfile.createDimension('longitude', longitude_len)

                    # --- Variables and their attributes ---
                    time = ncfile.createVariable('time', 'f8', ('time',))
                    time.units = 'seconds since 1970-01-01 00:00:00 UTC'
                    time.calendar = 'standard'
                    if debug:
                        print(f"Shape of time[:] {time[:].shape}")
                        print(f"Shape of unixtime {len(unixtime)}")
                    time[:] = unixtime

                    depth = ncfile.createVariable('depth', 'f4', ('depth',))
                    depth.units = 'meters'
                    depth[:] = mmdepth

                    latitude = ncfile.createVariable('latitude', 'f4', ('latitude',))
                    latitude.units = 'degrees_north'
                    latitude[:] = closest_lat_model

                    longitude = ncfile.createVariable('longitude', 'f4', ('longitude',))
                    longitude.units = 'degrees_east'
                    longitude[:] = closest_lon_model                     

                    salt = ncfile.createVariable('Salinity', 'f4', ('time', 'depth', 'latitude', 'longitude',))
                    salt.units = '"0.001'
                    salt.long_name = 'Practical salinity'
                    
                    final_salt_data = np.vstack(closestS)
                    salt[:, :] = final_salt_data                    

                print("NetCDF file created successfully.")                
                

            elif len(variable_values) == 2:   
                
                if var == '1':

                    # Extract climatology values for temperature
                    print("**************************************************************************************")
                    print(f"Start extracting Uliege climatology temperature data closest to {platform_code} location:")

                    if debug:
                        print(f"counter: {counter}") 
                
                    fileout = "climuliege-data_"+platform_code+"_sea_water_temperature.nc"
                    outdir = inclimdirT
                    output_filename = outdir+"/"+fileout

                    tindex_start = int(start_month) - 1
                    tindex_end = int(end_month)
                    if debug:
                        print(f"tindex_start: {tindex_start}")
                        print(f"tindex_end: {tindex_end}")                   
                
                    closestT = mm3dtemperature[tindex_start:tindex_end, :, j_idx, i_idx]
                    unixtime = seconds_since_1970[tindex_start:tindex_end]
                
                    if debug:
                        print(f"Shape of closestT {closestT.shape}")

                    # --- Write to NetCDF file ---
                    print(f"Creating: {output_filename}")
 
                    with Dataset(output_filename, 'w', format='NETCDF4') as ncfile:
                        # --- Global Attributes ---
                        # Copy attributes from the source file
                        for attr_name, attr_value in source_global_attrs_T.items():
                            ncfile.setncattr(attr_name, attr_value)
                       
                        # --- Dimensions ---
                        time_len = tindex_end - tindex_start
                        if debug:
                            print(f"time_len: {time_len}")
                        depth_len = len(mmdepth)
                        latitude_len = 1
                        longitude_len = 1                        
                        ncfile.createDimension('time', time_len)
                        ncfile.createDimension('depth', depth_len)
                        ncfile.createDimension('latitude', latitude_len)
                        ncfile.createDimension('longitude', longitude_len)
                        
                        # --- Variables and their attributes ---
                        time = ncfile.createVariable('time', 'f8', ('time',))
                        time.units = 'seconds since 1970-01-01 00:00:00 UTC'
                        time.calendar = 'standard'
                        if debug:
                            print(f"Shape of time[:] {time[:].shape}")
                            print(f"Shape of unixtime {len(unixtime)}")
                        time[:] = unixtime

                        depth = ncfile.createVariable('depth', 'f4', ('depth',))
                        depth.units = 'meters'
                        depth[:] = mmdepth

                        latitude = ncfile.createVariable('latitude', 'f4', ('latitude',))
                        latitude.units = 'degrees_north'
                        latitude[:] = closest_lat_model

                        longitude = ncfile.createVariable('longitude', 'f4', ('longitude',))
                        longitude.units = 'degrees_east'
                        longitude[:] = closest_lon_model  
                        
                        temp = ncfile.createVariable('Temperature', 'f4', ('time', 'depth', 'latitude', 'longitude',))
                        temp.units = 'celsius'
                        temp.long_name = 'Sea temperature'
                    
                        final_temp_data = np.vstack(closestT)
                        temp[:, :] = final_temp_data

                    print("NetCDF file created successfully.")          

                elif var == '2':

                    # Extract climatology values for salinity
                    print("**************************************************************************************")
                    print(f"Start extracting Uliege climatology salinity data closest to {platform_code} location:")

                    if debug:
                        print(f"counter: {counter}") 
                
                    fileout = "climuliege-data_"+platform_code+"_sea_water_practical_salinity.nc"
                    outdir = inclimdirS
                    output_filename = outdir+"/"+fileout

                    tindex_start = int(start_month) - 1
                    tindex_end = int(end_month)
                    if debug:
                        print(f"tindex_start: {tindex_start}")
                        print(f"tindex_end: {tindex_end}")                   
                
                    closestS = mm3dsalinity[tindex_start:tindex_end, :, j_idx, i_idx]
                    unixtime = seconds_since_1970[tindex_start:tindex_end]
                
                    if debug:
                        print(f"Shape of closestS {closestS.shape}")

                    # --- Write to NetCDF file ---
                    print(f"Creating: {output_filename}")
 
                    with Dataset(output_filename, 'w', format='NETCDF4') as ncfile:
                        # --- Global Attributes ---
                        # Copy attributes from the source file
                        for attr_name, attr_value in source_global_attrs_S.items():
                            ncfile.setncattr(attr_name, attr_value)
                       
                        # --- Dimensions ---
                        time_len = tindex_end - tindex_start
                        if debug:
                            print(f"time_len: {time_len}")
                        depth_len = len(mmdepth)
                        latitude_len = 1
                        longitude_len = 1                        
                        ncfile.createDimension('time', time_len)
                        ncfile.createDimension('depth', depth_len)
                        ncfile.createDimension('latitude', latitude_len)
                        ncfile.createDimension('longitude', longitude_len)                        

                        # --- Variables and their attributes ---
                        time = ncfile.createVariable('time', 'f8', ('time',))
                        time.units = 'seconds since 1970-01-01 00:00:00 UTC'
                        time.calendar = 'standard'
                        if debug:
                            print(f"Shape of time[:] {time[:].shape}")
                            print(f"Shape of unixtime {len(unixtime)}")
                        time[:] = unixtime

                        depth = ncfile.createVariable('depth', 'f4', ('depth',))
                        depth.units = 'meters'
                        depth[:] = mmdepth

                        latitude = ncfile.createVariable('latitude', 'f4', ('latitude',))
                        latitude.units = 'degrees_north'
                        latitude[:] = closest_lat_model

                        longitude = ncfile.createVariable('longitude', 'f4', ('longitude',))
                        longitude.units = 'degrees_east'
                        longitude[:] = closest_lon_model                          

                        salt = ncfile.createVariable('Salinity', 'f4', ('time', 'depth', 'latitude', 'longitude',))
                        salt.units = '"0.001'
                        salt.long_name = 'Practical salinity'
                    
                        final_salt_data = np.vstack(closestS)
                        salt[:, :] = final_salt_data                    

                    print("NetCDF file created successfully.")   

            counter += 1      

***

Vertically interpolate temperature and salinity time series on mooring's depths and derive daily values from monthly climatological values for the period defined by targeted_range variable

In [ ]:
# Vertically interpolate temperature and salinity time series on mooring's depths
# Use SOURCE function vertical_interpolation(in_file, depth_array_str, out_file, verbose)
# => vertical_interpolation.vertical_interpolation(location_ported_file, depth_array_str,
#                                                  vertical_interpolated_file, verbose=verbose)

data_type = 'MO'
frequency = 'dm'

# Define paths
outobsdir = parent_dir+'/output_NA/OBSERVATION/output/statistic/'+frequency
outobsdirT = outobsdir+'/sea_water_temperature/'
outobsdirS = outobsdir+'/sea_water_practical_salinity/'

workclimdir = parent_dir+'/output_NA/CLIM/output/work/'
if not os.path.exists(workclimdir):
    os.makedirs(workclimdir)


# Temperature
print(f"**== Processing temperature ==**")
sorted_files_obsT = sorted([ x for x in os.listdir(outobsdirT) if x.endswith('nc')])  
sorted_files_climT = sorted([ x for x in os.listdir(inclimdirT) if x.endswith('nc') and x.startswith('climuliege')])  

for f in sorted_files_obsT:
    mooring = f.split('_')[1]
    print(f"** Reading Platform: {mooring}")
    try:
        clim_file = [clim_f for clim_f in sorted_files_climT if clim_f.split('_')[1] == mooring][0]
        print(f"Corresponding clim file: {clim_file}")

        obs_nc = Dataset(outobsdirT+'/'+f,'r')
        obs_depth = obs_nc.variables['depth'][:]
        if debug:
            print(f"obs_depth: {obs_depth}")
        obs_depth_str = ' '.join([str(value) for value in obs_depth])

        clim_nc = Dataset(inclimdirT+'/'+clim_file,'r')
        clim_depth = clim_nc.variables['depth'][:]
        if debug:
            print(f"clim_depth: {clim_depth}")

        infileprefix = clim_file.split('-')[0]
        infilesuffix = '-'.join(clim_file.split('-')[1:])
        outfileprefix = "climuliegeint"
        outfilename =  outfileprefix + '-' + infilesuffix
        print(f"Output file name: {outfilename}")

        vertical_interpolation.vertical_interpolation(inclimdirT+clim_file, obs_depth_str,
                                                      workclimdir+outfilename, True)
    except IndexError as e:
        msg = f"Climatological temperature file not found for platform {mooring}"
        print(msg)
        continue


# Salinity
print(f"**== Processing salinity ==**")
sorted_files_obsS = sorted([ x for x in os.listdir(outobsdirS) if x.endswith('nc')])  
sorted_files_climS = sorted([ x for x in os.listdir(inclimdirS) if x.endswith('nc') and x.startswith('climuliege')])

for f in sorted_files_obsS:
    mooring = f.split('_')[1]
    print(f"** Reading Platform: {mooring}")
    try:
        clim_file = [clim_f for clim_f in sorted_files_climS if clim_f.split('_')[1] == mooring][0]
        print(f"Corresponding clim file: {clim_file}")

        obs_nc = Dataset(outobsdirS+'/'+f,'r')
        obs_depth = obs_nc.variables['depth'][:]
        if debug:
            print(f"obs_depth: {obs_depth}")
        obs_depth_str = ' '.join([str(value) for value in obs_depth])

        clim_nc = Dataset(inclimdirS+'/'+clim_file,'r')
        clim_depth = clim_nc.variables['depth'][:]
        if debug:
            print(f"clim_depth: {clim_depth}")

        infileprefix = clim_file.split('-')[0]
        infilesuffix = '-'.join(clim_file.split('-')[1:])
        outfileprefix = "climuliegeint"
        outfilename =  outfileprefix + '-' + infilesuffix
        print(f"Output file name: {outfilename}")

        vertical_interpolation.vertical_interpolation(inclimdirS+clim_file, obs_depth_str,
                                                  workclimdir+outfilename, True)
    except IndexError as e:
        msg = f"Climatological salinity file not found for platform {mooring}"
        print(msg)
        continue        

***

Derive daily values from monthly climatological values for the period defined by targeted_range variable

In [ ]:
from datetime import datetime, timedelta

debug = False

outclimdir = parent_dir+'/output_NA/CLIM/output/statistic/'+frequency
outclimdirT = outclimdir+'/sea_water_temperature/'
outclimdirS = outclimdir+'/sea_water_practical_salinity/'

if not os.path.exists(outclimdirT):
    os.makedirs(outclimdirT)    

if not os.path.exists(outclimdirS):
    os.makedirs(outclimdirS)   


def derive_daily_values_constant(targeted_range, monthly_values):
    """
    Derives daily values from monthly climatological values using a constant
    assignment method.

    Args:
        targeted_range (str): A string representing the date range in the format
                              'YYYY-MM-DDTHH:MM:SSZ/YYYY-MM-DDTHH:MM:SSZ'.
        monthly_values (dict): A dictionary where keys are month numbers (1-12)
                               and values are the climatological values for that month.

    Returns:
        pandas.DataFrame: A DataFrame with two columns: 'date' and 'value',
                          containing the daily derived values.
    """
    # --- 1. Parse the targeted date range ---
    try:
        start_date_str, end_date_str = targeted_range.split('/')
        # Parse the ISO 8601 format, ignoring the 'Z' for UTC
        start_date = datetime.fromisoformat(start_date_str.replace('Z', '+00:00'))
        #print(f"Start date {start_date}.")
        end_date = datetime.fromisoformat(end_date_str.replace('Z', '+00:00'))
        #print(f"End date {end_date}.")        
    except ValueError as e:
        print(f"Error parsing date range: {e}")
        return None

    # --- 2. Generate a list of dates for the period ---
    # Use pandas date_range for convenience
    daily_dates = pd.date_range(start=start_date, end=end_date, freq='D')

    # --- 3. Assign monthly values to each day ---
    derived_data = []
    for date in daily_dates:
        month = date.month
        # Look up the value for the current month
        value = monthly_values.get(month)
        #print(f"value: {value}.")

        if value is not None:
            derived_data.append({'date': date, 'value': value})
        else:
            # Handle cases where a month's data might be missing
            print(f"Warning: No value found for month {month}. Skipping date {date.date()}.")


    # --- 4. Create and return a pandas DataFrame ---
    if not derived_data:
        return pd.DataFrame(columns=['date', 'value'])

    return pd.DataFrame(derived_data)


print(f"Deriving daily values for the period: {targeted_range}\n")

sorted_files_climT = sorted([ x for x in os.listdir(workclimdir) if x.endswith('temperature.nc') and x.startswith('climuliegeint')])
sorted_files_climS = sorted([ x for x in os.listdir(workclimdir) if x.endswith('salinity.nc') and x.startswith('climuliegeint')])


# Temperature
print(f"==== Processing temperature data ====")
for f in sorted_files_climT:
    mooring = f.split('_')[1]
    print(f"** Reading Platform: {mooring}")

    clim_nc = Dataset(workclimdir+'/'+f,'r')
    source_global_attrs = {attr: clim_nc.getncattr(attr) for attr in clim_nc.ncattrs()}
    clim_var = clim_nc.variables['Temperature'][:]
    clim_depth = clim_nc.variables['depth'][:]
    # 1. Get the time variable object from the file
    clim_time = clim_nc.variables['time']
    # 2. Now, get the numerical values from the object.
    clim_time_values = clim_time[:]
    # 3. Call num2date using the attributes from the variable object.
    datetime_objects = num2date(clim_time_values, 
                            units=clim_time.units, 
                            calendar=clim_time.calendar)
    # 4. Loop through the datetime objects to format and extract the month
    month_list = []
    for dt in datetime_objects:
        # Format the date as 'YYYY-MM-DD'
        formatted_date = dt.strftime('%Y-%m-%d')
        
        # Extract the month value
        month_value = dt.month

        # Append the extracted month to a list
        month_list.append(month_value)
        
        print(f"Original datetime: {dt}, Formatted Date: {formatted_date}, Extracted Month: {month_value}")
    
    clim_var_dict = {}
    for i in range(len(month_list)):
        month_number = month_list[i]
        temperature_profile = clim_var[i]
        clim_var_dict[month_number] = temperature_profile

    daily_values_df = derive_daily_values_constant(targeted_range, clim_var_dict)

    # Add a column with time in seconds since epoch ---
    # Define the Unix epoch start time (timezone-aware)
    epoch_start = pd.Timestamp("1970-01-01", tz='UTC')
    # Add 12 hours to the dates and then convert into seconds
    daily_values_df['date'] = daily_values_df['date'] + pd.Timedelta(hours=12)
    daily_values_df['time_seconds'] = (daily_values_df['date'].dt.tz_convert('UTC') - epoch_start).dt.total_seconds().astype(float)
    
    if debug: 
        # Set pandas to display all rows to see the full result
        pd.set_option('display.max_rows', None)
                
        print("--- Derived Daily Temperature Values (with seconds from epoch) ---")
        print(daily_values_df.to_string(
            formatters={'time_seconds': '{:.0f}'.format},
            index=False
        ))
        print("----------------------------------------------------------------")

    
    # --- Write to NetCDF file ---
    infileclim = workclimdir+'/'+f
    print(f"Input clim file: {infileclim}")
    infileprefix = infileclim.split('-')[0]
    infilesuffix = '-'.join(infileclim.split('-')[1:])
    outfileprefix = "climuliegedaily"
    output_nc_filename = outclimdirT + outfileprefix + '-' + infilesuffix
    print(f"Output clim file: {output_nc_filename}")

    with Dataset(output_nc_filename, 'w', format='NETCDF4') as ncfile:
        # --- Global Attributes ---
        # Copy attributes from the source file
        for attr_name, attr_value in source_global_attrs.items():
            ncfile.setncattr(attr_name, attr_value)
        # Add global attributes to the processed files
        ncfile.institution = 'University of Liège, GeoHydrodynamics and Environment Research'
        ncfile.SOURCE_institution = 'Istituto Nazionale di Geofisica e Vulcanologia - Bologna, Italy'
        ncfile.SOURCE_platform_code = mooring
        ncfile.SOURCE_variable_type = 'Daily values derived from monthly climatologies'
        ncfile.SOURCE_field_type = 'Vertical interpolated climatology data'
                       
        # --- Dimensions ---
        time_len = len(daily_values_df)
        depth_len = len(clim_depth)
        ncfile.createDimension('time', time_len)
        ncfile.createDimension('depth', depth_len)

        # --- Variables and their attributes ---
        time = ncfile.createVariable('time', 'f8', ('time',))
        time.units = 'seconds since 1970-01-01 00:00:00 UTC'
        time[:] = daily_values_df['time_seconds'].values

        depth = ncfile.createVariable('depth', 'f4', ('depth',))
        depth.units = 'meters'
        depth[:] = clim_depth
                
        temp = ncfile.createVariable('sea_water_temperature', 'f4', ('time', 'depth'))
        temp.units = 'celsius'
        temp.long_name = 'Sea temperature'
                
        final_temp_data = np.vstack(daily_values_df['value'].values)
        temp[:, :] = final_temp_data

    print("NetCDF file created successfully.")



# Salinity
print(f"==== Processing salinity data ====")
for f in sorted_files_climS:
    mooring = f.split('_')[1]
    print(f"** Reading Platform: {mooring}")

    clim_nc = Dataset(workclimdir+'/'+f,'r')
    source_global_attrs = {attr: clim_nc.getncattr(attr) for attr in clim_nc.ncattrs()}
    clim_var = clim_nc.variables['Salinity'][:]
    clim_depth = clim_nc.variables['depth'][:]
    # 1. Get the time variable object from the file
    clim_time = clim_nc.variables['time']
    # 2. Now, get the numerical values from the object.
    clim_time_values = clim_time[:]
    # 3. Call num2date using the attributes from the variable object.
    datetime_objects = num2date(clim_time_values, 
                            units=clim_time.units, 
                            calendar=clim_time.calendar)
    # 4. Loop through the datetime objects to format and extract the month
    month_list = []
    for dt in datetime_objects:
        # Format the date as 'YYYY-MM-DD'
        formatted_date = dt.strftime('%Y-%m-%d')
        
        # Extract the month value
        month_value = dt.month

        # Append the extracted month to a list
        month_list.append(month_value)
        
        print(f"Original datetime: {dt}, Formatted Date: {formatted_date}, Extracted Month: {month_value}")
    
    clim_var_dict = {}
    for i in range(len(month_list)):
        month_number = month_list[i]
        salinity_profile = clim_var[i]
        clim_var_dict[month_number] = salinity_profile

    daily_values_df = derive_daily_values_constant(targeted_range, clim_var_dict)

    # Add a column with time in seconds since epoch ---
    # Define the Unix epoch start time (timezone-aware)
    epoch_start = pd.Timestamp("1970-01-01", tz='UTC')
    # Add 12 hours to the dates and then convert into seconds
    daily_values_df['date'] = daily_values_df['date'] + pd.Timedelta(hours=12)
    daily_values_df['time_seconds'] = (daily_values_df['date'].dt.tz_convert('UTC') - epoch_start).dt.total_seconds().astype(float)
    
    if debug: 
        # Set pandas to display all rows to see the full result
        pd.set_option('display.max_rows', None)
                
        print("--- Derived Daily Salinity Values (with seconds from epoch) ---")
        print(daily_values_df.to_string(
            formatters={'time_seconds': '{:.0f}'.format},
            index=False
        ))
        print("----------------------------------------------------------------")

    
    # --- Write to NetCDF file ---
    infileclim = workclimdir+'/'+f
    print(f"Input clim file: {infileclim}")
    infileprefix = infileclim.split('-')[0]
    infilesuffix = '-'.join(infileclim.split('-')[1:])
    outfileprefix = "climuliegedaily"
    output_nc_filename = outclimdirS + outfileprefix + '-' + infilesuffix
    print(f"Output clim file: {output_nc_filename}")

    with Dataset(output_nc_filename, 'w', format='NETCDF4') as ncfile:
        # --- Global Attributes ---
        # Copy attributes from the source file
        for attr_name, attr_value in source_global_attrs.items():
            ncfile.setncattr(attr_name, attr_value)
        # Add global attributes to the processed files
        ncfile.institution = 'University of Liège, GeoHydrodynamics and Environment Research'
        ncfile.SOURCE_institution = 'Istituto Nazionale di Geofisica e Vulcanologia - Bologna, Italy'
        ncfile.SOURCE_platform_code = mooring
        ncfile.SOURCE_variable_type = 'Daily values derived from monthly climatologies'
        ncfile.SOURCE_field_type = 'Vertical interpolated climatology data'
                       
        # --- Dimensions ---
        time_len = len(daily_values_df)
        depth_len = len(clim_depth)
        ncfile.createDimension('time', time_len)
        ncfile.createDimension('depth', depth_len)

        # --- Variables and their attributes ---
        time = ncfile.createVariable('time', 'f8', ('time',))
        time.units = 'seconds since 1970-01-01 00:00:00 UTC'
        time[:] = daily_values_df['time_seconds'].values

        depth = ncfile.createVariable('depth', 'f4', ('depth',))
        depth.units = 'meters'
        depth[:] = clim_depth
                
        salt = ncfile.createVariable('sea_water_practical_salinity', 'f4', ('time', 'depth'))
        salt.units = '"0.001'
        salt.long_name = 'Practical salinity'
                
        final_salt_data = np.vstack(daily_values_df['value'].values)
        salt[:, :] = final_salt_data

    print("NetCDF file created successfully.")